# Hestia Wake Word Training (OpenWakeWord)

Trains a custom wake word model using synthetic TTS clips + optional scraped real audio.

**Steps:**
1. Test pronunciation of target word
2. Download training data (background noise, RIRs, features)
3. Mount Drive & unzip scraped data
4. Generate synthetic clips
5. Inject scraped positives
6. Augment + Train + Download

In [ ]:
# @title 1. Test Example Training Clip Generation { display-mode: "form" }
# @markdown Type in your target wake word below and run to hear how it sounds.
# @markdown Tips:
# @markdown - Spell phonetically with underscores if mispronounced: "artemis" → "ar_teh_miss"
# @markdown - Spell out numbers ("2" → "two")
# @markdown - Avoid punctuation except "?" and "!"

import os
import sys
from IPython.display import Audio

if not os.path.exists("./piper-sample-generator"):
    !git clone https://github.com/rhasspy/piper-sample-generator
    !wget -O piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
    !cd piper-sample-generator && git checkout 213d4d5

    # Install system dependencies
    !pip install piper-tts piper-phonemize-cross
    !pip install webrtcvad
    !pip install torch==2.5.0 torchvision==0.20.0 torchaudio==2.5.0 --index-url https://download.pytorch.org/whl/cu121

target_word = 'artemis' # @param {type:"string"}

if "piper-sample-generator/" not in sys.path:
    sys.path.append("piper-sample-generator/")
from generate_samples import generate_samples

def text_to_speech(text):
    generate_samples(text = text,
                max_samples=1,
                length_scales=[1.1],
                noise_scales=[0.7], noise_scale_ws = [0.7],
                output_dir = './', batch_size=1, auto_reduce_batch_size=True,
                file_names=["test_generation.wav"]
                )

text_to_speech(target_word)

from IPython.display import Audio
Audio("test_generation.wav", autoplay=True)

In [ ]:
# @title 2. Download Data { display-mode: "form" }
# @markdown Downloads background noise, music, RIRs, and pre-computed features.
# @markdown Takes ~15 minutes.

import locale
def getpreferredencoding(do_setlocale = True):
    return "UTF-8"
locale.getpreferredencoding = getpreferredencoding

# install openwakeword (full installation to support training)
!git clone https://github.com/dscripka/openwakeword
!pip install -e ./openwakeword --no-deps

# install other dependencies
!pip install mutagen==1.47.0
!pip install torchinfo==1.8.0
!pip install torchmetrics==1.2.0
!pip install speechbrain==0.5.14
!pip install audiomentations==0.33.0
!pip install torch-audiomentations==0.11.0
!pip install acoustics==0.2.6
!pip install onnxruntime==1.22.1 ai_edge_litert==1.4.0 onnxsim
!pip install onnx2tf
!pip install onnx==1.19.1
!pip install onnx_graphsurgeon
!pip install sng4onnx
!pip install pronouncing==0.2.0
!pip install datasets==2.14.6
!pip install deep-phonemizer==0.0.19

# Download required models
import os
os.makedirs("./openwakeword/openwakeword/resources/models", exist_ok=True)
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx -O ./openwakeword/openwakeword/resources/models/embedding_model.onnx
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.tflite -O ./openwakeword/openwakeword/resources/models/embedding_model.tflite
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx -O ./openwakeword/openwakeword/resources/models/melspectrogram.onnx
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.tflite -O ./openwakeword/openwakeword/resources/models/melspectrogram.tflite

# Imports
import sys

if "piper-sample-generator/" not in sys.path:
    sys.path.append("piper-sample-generator/")
from generate_samples import generate_samples

import numpy as np
import torch
import sys
from pathlib import Path
import uuid
import yaml
import datasets
import scipy
from tqdm import tqdm

## Download MIR RIR data (~2 minutes)
output_dir = "./mit_rirs"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
    !git lfs install
    !git clone https://huggingface.co/datasets/davidscripka/MIT_environmental_impulse_responses
    rir_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("./MIT_environmental_impulse_responses/16khz").glob("*.wav")]}).cast_column("audio", datasets.Audio())
    for row in tqdm(rir_dataset):
        name = row['audio']['path'].split('/')[-1]
        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

## Download noise and background audio (~3 minutes)
if not os.path.exists("audioset"):
    os.mkdir("audioset")

    fname = "bal_train09.tar"
    out_dir = f"audioset/{fname}"
    link = "https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/" + fname
    !wget -O {out_dir} {link}
    !cd audioset && tar -xvf bal_train09.tar

    output_dir = "./audioset_16k"
    if not os.path.exists(output_dir):
        os.mkdir(output_dir)

    audioset_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("audioset/audio").glob("**/*.flac")]})
    audioset_dataset = audioset_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
    for row in tqdm(audioset_dataset):
        name = row['audio']['path'].split('/')[-1].replace(".flac", ".wav")
        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

# Free Music Archive dataset
output_dir = "./fma"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
    fma_dataset = datasets.load_dataset("rudraml/fma", name="small", split="train", streaming=True)
    fma_dataset = iter(fma_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000)))

    n_hours = 1
    for i in tqdm(range(n_hours*3600//30)):
        row = next(fma_dataset)
        name = row['audio']['path'].split('/')[-1].replace(".mp3", ".wav")
        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))
        i += 1
        if i == n_hours*3600//30:
            break

# Download pre-computed features
if not os.path.exists("./openwakeword_features_ACAV100M_2000_hrs_16bit.npy"):
    !wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy

if not os.path.exists("validation_set_features.npy"):
    !wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy

In [ ]:
# @title 3. Mount Drive & Unzip Scraped Data { display-mode: "form" }
# @markdown Mounts Google Drive and extracts the scraped training data.
# @markdown 
# @markdown Set the word and Drive path below:

wake_word = 'artemis' # @param {type:"string"}
drive_zip_path = '/content/drive/My Drive/Wakeword/training_data_artemis.zip' # @param {type:"string"}

from google.colab import drive
drive.mount('/content/drive')

import os
if os.path.exists(drive_zip_path):
    !unzip -o "{drive_zip_path}" -d /content/
    # Count what we got
    scrape_dir = f'/content/training_data_{wake_word}'
    if os.path.exists(scrape_dir):
        pos = len([f for f in os.listdir(f'{scrape_dir}/positive') if f.endswith('.wav')]) if os.path.exists(f'{scrape_dir}/positive') else 0
        neg = len([f for f in os.listdir(f'{scrape_dir}/negative') if f.endswith('.wav')]) if os.path.exists(f'{scrape_dir}/negative') else 0
        hard = len([f for f in os.listdir(f'{scrape_dir}/hard_negative') if f.endswith('.wav')]) if os.path.exists(f'{scrape_dir}/hard_negative') else 0
        print(f'\nScraped data for {wake_word}:')
        print(f'  Positive:      {pos}')
        print(f'  Negative:      {neg}')
        print(f'  Hard negative: {hard}')
    else:
        print(f'WARNING: {scrape_dir} not found after unzip')
else:
    print(f'WARNING: {drive_zip_path} not found — training will use synthetic clips only')

In [ ]:
# @title 4. Generate Synthetic Clips { display-mode: "form" }
# @markdown Generates synthetic TTS clips of the wake word using piper.

import yaml
import sys

number_of_examples = 1000 # @param {type:"slider", min:100, max:50000, step:50}
number_of_training_steps = 10000  # @param {type:"slider", min:0, max:50000, step:100}
false_activation_penalty = 1500  # @param {type:"slider", min:100, max:5000, step:50}

config = yaml.load(open("openwakeword/examples/custom_model.yml", 'r').read(), yaml.Loader)

config["target_phrase"] = [target_word]
config["model_name"] = config["target_phrase"][0].replace(" ", "_")
config["n_samples"] = number_of_examples
config["n_samples_val"] = max(500, number_of_examples//10)
config["steps"] = number_of_training_steps
config["target_accuracy"] = 0.5
config["target_recall"] = 0.25
config["output_dir"] = "./my_custom_model"
config["max_negative_weight"] = false_activation_penalty

config["background_paths"] = ['./audioset_16k', './fma']
config["false_positive_validation_data_path"] = "validation_set_features.npy"
config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}

with open('my_model.yaml', 'w') as file:
    yaml.dump(config, file)

# Generate synthetic clips only
!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --generate_clips

# Show what was generated
import os
clip_dir = f"my_custom_model/{config['model_name']}/positive_train"
if os.path.exists(clip_dir):
    count = len([f for f in os.listdir(clip_dir) if f.endswith('.wav')])
    print(f'\nSynthetic clips generated: {count} in {clip_dir}/')
else:
    print(f'\nWARNING: Expected directory {clip_dir} not found.')
    print('Checking actual structure:')
    !find my_custom_model -name "*.wav" -type f | head -10

In [ ]:
# @title 5. Inject Scraped Positives { display-mode: "form" }
# @markdown Copies real scraped audio clips into the synthetic clips directory
# @markdown so they get augmented and trained alongside the TTS clips.

import os
import shutil

model_name = config['model_name']
clip_dir = f"my_custom_model/{model_name}/positive_train"
scrape_pos = f"/content/training_data_{wake_word}/positive"

if not os.path.exists(clip_dir):
    # Try to find the actual clip directory
    import subprocess
    result = subprocess.run(['find', 'my_custom_model', '-name', '*.wav', '-type', 'f'],
                          capture_output=True, text=True)
    if result.stdout.strip():
        first_file = result.stdout.strip().split('\n')[0]
        clip_dir = os.path.dirname(first_file)
        print(f'Found clips at: {clip_dir}/')
    else:
        print('ERROR: No generated clips found. Run Step 4 first.')

if os.path.exists(clip_dir) and os.path.exists(scrape_pos):
    before = len([f for f in os.listdir(clip_dir) if f.endswith('.wav')])

    # Copy scraped positives
    scraped_files = [f for f in os.listdir(scrape_pos) if f.endswith('.wav')]
    for f in scraped_files:
        shutil.copy2(os.path.join(scrape_pos, f), os.path.join(clip_dir, f'scraped_{f}'))

    after = len([f for f in os.listdir(clip_dir) if f.endswith('.wav')])
    print(f'Before: {before} clips (synthetic)')
    print(f'Added:  {len(scraped_files)} clips (scraped)')
    print(f'Total:  {after} clips ready for augmentation')
elif not os.path.exists(scrape_pos):
    print(f'No scraped data at {scrape_pos} — training with synthetic clips only')
    print('(This is fine, but scraped data improves real-world accuracy)')

In [ ]:
# @title 6. Augment + Train + Download { display-mode: "form" }
# @markdown Augments all clips (synthetic + scraped), trains the model,
# @markdown converts to tflite, and downloads both formats.
# @markdown
# @markdown ~30-60 min on CPU runtime. Use GPU runtime for faster training.

import sys

# Augment clips
!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --augment_clips

# Train model
!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --train_model

# Convert ONNX to tflite
onnx_model_path = f"my_custom_model/{config['model_name']}.onnx"
name1 = f"my_custom_model/{config['model_name']}_float32.tflite"
name2 = f"my_custom_model/{config['model_name']}.tflite"
!onnx2tf -i {onnx_model_path} -o my_custom_model/ -kat onnx____Flatten_0
!mv {name1} {name2}

# Download trained models
from google.colab import files
files.download(f"my_custom_model/{config['model_name']}.onnx")
files.download(f"my_custom_model/{config['model_name']}.tflite")

print(f'\nDone! Model saved as: {config["model_name"]}.onnx')
print(f'Deploy to: models/{config["model_name"]}.onnx in the Hestia repo')